# Kimi K2.5 Linear FLOPs Constant

This notebook converts `kimi25_linear_flops_constant.html` into a block-by-block calculation notebook.

Target result:

`linear_flops_per_token = 61,023,322,112`

Scope contract: single giant GPU, no parallelism, multiply-add counted as 2 FLOPs, Kimi K2.5 active transformer-block math. Router is included. Attention scan, softmax, RMSNorm, RoPE elementwise work, sampling, communication, and `lm_head` are excluded.

## 1. Model constants

These names intentionally match the HTML pages and the full-model architecture map.

In [ ]:
H = 7168
L = 61
num_attention_heads = 64
heads = num_attention_heads

q_lora = 1536
kv_lora = 512
qk_nope = 128
qk_rope = 64
qk_head = qk_nope + qk_rope
v_head = 128

dense_intermediate = 18432
moe_intermediate = 2048
dense_layers = 1
moe_layers = 60
routed_experts = 384
topk = 8
shared_experts = 1
active_experts = topk + shared_experts

assert heads == 64
assert qk_head == 192
assert dense_layers + moe_layers == L
assert active_experts == 9

constants = {
    'H': H,
    'L': L,
    'heads': heads,
    'q_lora': q_lora,
    'kv_lora': kv_lora,
    'qk_nope': qk_nope,
    'qk_rope': qk_rope,
    'qk_head': qk_head,
    'v_head': v_head,
    'dense_intermediate': dense_intermediate,
    'moe_intermediate': moe_intermediate,
    'routed_experts': routed_experts,
    'active_experts': active_experts,
}
constants

## 2. Attention projection FLOPs per layer

This is still part of the **per-token linear FLOPs** constant because it is projection work done once per forward token.

Components:

- `qkv_a`: one full projection on the single giant GPU.
- `q_b`: all 64 heads are included.
- absorbed `W_KC`: maps the `qk_nope=128` query side into latent `kv_lora=512` space before scan.
- absorbed `W_VC`: maps latent attention output back toward `v_head=128` after scan.
- `o_proj`: all attention-output dimensions are included.

### 2.1 Parse the article equations and link them to our FLOP equations

The screenshot from the Zhihu article is equation (37)-(47). It is describing MLA dataflow, not FLOP counting. Below is the same flow parsed into our variable names.

#### Article equations, translated

| Article eq | Article formula | Meaning | Kimi variable in this notebook | FLOP term that counts the projection |
| --- | --- | --- | --- | --- |
| (37) | `c_t^Q = W^DQ h_t` | Compress hidden state into a query latent. | `q_lora = 1536` | Part of `qkv_a = 2 * H * (...)` |
| (38) | `[q^C_{t,1}, ..., q^C_{t,n_h}] = q_t^C = W^UQ c_t^Q` | Expand query latent into per-head **content/no-RoPE query**. | `heads * qk_nope = 64 * 128` | Content part of `q_b = 2 * q_lora * (heads * qk_head)` |
| (39) | `[q^R_{t,1}, ..., q^R_{t,n_h}] = q_t^R = RoPE(W^QR c_t^Q)` | Expand query latent into per-head **RoPE query** and apply RoPE. | `heads * qk_rope = 64 * 64` | RoPE part of `q_b = 2 * q_lora * (heads * qk_head)`; RoPE elementwise op itself is not counted |
| (40) | `q_{t,i} = [q^C_{t,i}; q^R_{t,i}]` | Concatenate content query and RoPE query for head `i`. | `qk_head = qk_nope + qk_rope = 192` | No matmul here; this explains why `q_b` outputs `heads * qk_head` |
| (41) | `c_t^KV = W^DKV h_t` | Compress hidden state into latent KV cache vector. | `kv_lora = 512` | Part of `qkv_a = 2 * H * (...)` |
| (42) | `[k^C_{t,1}, ..., k^C_{t,n_h}] = k_t^C = W^UK c_t^KV` | Expand latent KV into per-head content keys. With matrix absorption, SGLang does not need to materialize/cache these full keys. | `qk_nope = 128`, `kv_lora = 512` | Counted as absorbed `W_KC = 2 * heads * qk_nope * kv_lora` |
| (43) | `k_t^R = RoPE(W^KR h_t)` | Create the small shared RoPE key branch directly from hidden state. | `qk_rope = 64` | Part of `qkv_a = 2 * H * (...)`; RoPE elementwise op itself is not counted |
| (44) | `k_{t,i} = [k^C_{t,i}; k_t^R]` | Concatenate per-head content key and shared RoPE key. | key dimension would be `qk_nope + qk_rope` | No separate matmul; this explains the attention score has content part plus RoPE part |
| (45) | `[v^C_{t,1}, ..., v^C_{t,n_h}] = v_t^C = W^UV c_t^KV` | Expand latent KV into per-head values. With absorption, SGLang can accumulate latent values then project afterward. | `kv_lora = 512`, `v_head = 128` | Counted as absorbed `W_VC = 2 * heads * kv_lora * v_head` |
| (46) | `o_{t,i} = sum_j Softmax_i(q^T_{t,i} k_{j,i} / sqrt(d_h + d_h^R)) v^C_{j,i}` | Actual attention scan for head `i`: QK scores and AV accumulation over context. | `kv_lora`, `qk_rope`, `heads` | **Not** in this linear notebook; counted in `attention_scan_flops_per_pair` in the pair notebook |
| (47) | `u_t = W^O [o_{t,1}; ...; o_{t,n_h}]` | Project concatenated head outputs back to hidden size. | `heads * v_head -> H` | `o_proj = 2 * (heads * v_head) * H` |

#### Now the notebook equations become easier

`qkv_a = 2 * H * (q_lora + kv_lora + qk_rope)`

This combines article eq (37), eq (41), and eq (43): from `h_t[H]`, produce query latent `c_t^Q[q_lora]`, KV latent `c_t^KV[kv_lora]`, and RoPE key branch `k_t^R[qk_rope]`.

`q_b = 2 * q_lora * (heads * qk_head)`

This combines article eq (38)-(40): expand `c_t^Q[q_lora]` into all per-head query vectors, where each head has `qk_nope + qk_rope = qk_head`.

`W_KC = 2 * heads * qk_nope * kv_lora`

This corresponds to article eq (42), but in absorbed form. Instead of materializing all `k^C_{t,i}`, SGLang absorbs the key expansion into the query side, so the scan can compare a `512`-dim absorbed query with cached `c_j^KV[512]`.

`W_VC = 2 * heads * kv_lora * v_head`

This corresponds to article eq (45), also in absorbed form. Instead of materializing full per-head values first, the scan accumulates latent `c_j^KV[512]`, then projects toward `v_head=128`.

`o_proj = 2 * (heads * v_head) * H`

This is article eq (47): after attention produces all head outputs, concatenate them and project back to hidden size `H`.

The key mental split: article eq (46) is the attention scan over context and belongs to the separate pair notebook. The equations in this cell count only the per-token projection matmuls around MLA.

Article source: https://zhuanlan.zhihu.com/p/16730036197


In [3]:
# Article link: hidden h_t is projected into MLA latent pieces.
# Kimi fuses query latent q_lora, KV latent c_t^KV (kv_lora), and small RoPE key branch.
qkv_a = 2 * H * (q_lora + kv_lora + qk_rope)

# Article link: Q also uses a low-rank path, then expands to per-head query dimensions.
q_b = 2 * q_lora * (heads * qk_head)

# Article link: matrix absorption for the content/no-RoPE K path.
# Instead of materializing full k_j^C, compare absorbed q_content[512] with cached c_j^KV[512].
W_KC = 2 * heads * qk_nope * kv_lora

# Article link: V expansion is also absorbed; cache latent c_j^KV, project after latent AV accumulation.
W_VC = 2 * heads * kv_lora * v_head

# Standard attention output projection back to hidden size H.
o_proj = 2 * (heads * v_head) * H

attention_projection_per_layer = qkv_a + q_b + W_KC + W_VC + o_proj

assert qkv_a == 30_277_632
assert q_b == 37_748_736
assert W_KC == 8_388_608
assert W_VC == 8_388_608
assert o_proj == 117_440_512
assert attention_projection_per_layer == 202_244_096

{
    'qkv_a': qkv_a,
    'q_b': q_b,
    'W_KC': W_KC,
    'W_VC': W_VC,
    'o_proj': o_proj,
    'attention_projection_per_layer': attention_projection_per_layer,
}

{'qkv_a': 30277632,
 'q_b': 37748736,
 'W_KC': 8388608,
 'W_VC': 8388608,
 'o_proj': 117440512,
 'attention_projection_per_layer': 202244096}

## 3. `kv_b_proj` vs absorbed `W_KC/W_VC`: equivalence check

Important distinction: this section is an **accounting equivalence**, not a claim that the optimized absorbed path literally runs one fused `kv_b_proj` matmul at inference time.

The model class defines `kv_b_proj` with output shape:

`kv_lora -> heads * (qk_nope + v_head)`

Conceptually, that is the article's key/value expansion from latent `c_t^KV` into per-head content keys and values. But in SGLang's absorbed MLA forward path, the useful pieces are exposed and executed as two separate absorbed weights:

- `w_kc`: used on the query side as `q_nope @ w_kc`, corresponding to the content-key part.
- `w_vc`: used after latent attention accumulation as `attn_output @ w_vc`, corresponding to the value part.

So the FLOPs should be identical as a bookkeeping check:

`2 * kv_lora * (heads * (qk_nope + v_head)) == W_KC + W_VC`

Read this as: the full conceptual `kv_b_proj` has the same arithmetic size as the two absorbed pieces. Do **not** read it as: the absorbed runtime path first computes full `kv_b_proj` and then splits it.


In [ ]:
kv_b_equivalent = 2 * kv_lora * (heads * (qk_nope + v_head))
absorbed_split_equivalent = W_KC + W_VC

assert kv_b_equivalent == 16_777_216
assert absorbed_split_equivalent == 16_777_216
assert kv_b_equivalent == absorbed_split_equivalent

{
    'kv_b_equivalent': kv_b_equivalent,
    'W_KC_plus_W_VC': absorbed_split_equivalent,
}

## 4. MLP, MoE, and router FLOPs

SwiGLU has gate, up, and down projections, so the dense matmul cost is `6 * hidden * intermediate`.

For Kimi K2.5:

- Layer 0 is dense: `dense_layers = 1`.
- Layers 1-60 are MoE: `moe_layers = 60`.
- Each token runs `topk=8` routed experts plus `shared_experts=1`, so `active_experts=9`.
- Router scores all `routed_experts=384` routed experts.

### 4.1 Raw equations for Dense MLP, MoE, and router    

We do not have a Zhihu-style equation screenshot for this part in the current notes. The closest existing visual explanation is `kimi25_full_model_architecture.html`, section `Zoom-In: MoE Variables`, which shows router -> top-k routed experts -> shared expert -> expert MLP. Below is the same idea written as raw equations.

#### Dense layer 0: ordinary SwiGLU MLP

For layer 0, Kimi uses a dense SwiGLU MLP, not MoE:

```text
g_t = x_t W_gate              # [H] -> [dense_intermediate]
u_t = x_t W_up                # [H] -> [dense_intermediate]
m_t = silu(g_t) ⊙ u_t         # elementwise, ignored in this FLOP count
y_t = m_t W_down              # [dense_intermediate] -> [H]
```

Matmul FLOPs:

```text
dense_mlp_layer
= 2*H*dense_intermediate      # gate
+ 2*H*dense_intermediate      # up
+ 2*dense_intermediate*H      # down
= 6*H*dense_intermediate
```

#### MoE layers 1-60: router plus active experts

For each MoE layer, the router first scores all routed experts:

```text
router_logits_t = x_t W_router        # [H] -> [routed_experts]
selected_t = topk(router_logits_t, k=8)
```

Router matmul FLOPs:

```text
router_per_moe_layer = 2*H*routed_experts
```

Then only the active experts run. For Kimi K2.5:

```text
topk = 8
shared_experts = 1
active_experts = topk + shared_experts = 9
```

For one active expert `e`, the expert is also a SwiGLU MLP:

```text
g_{t,e} = x_t W_gate,e           # [H] -> [moe_intermediate]
u_{t,e} = x_t W_up,e             # [H] -> [moe_intermediate]
m_{t,e} = silu(g_{t,e}) ⊙ u_{t,e}
y_{t,e} = m_{t,e} W_down,e       # [moe_intermediate] -> [H]
```

One expert matmul FLOPs:

```text
one_expert = 6*H*moe_intermediate
```

All active expert FLOPs for one token in one MoE layer:

```text
active_moe_layer = active_experts * 6*H*moe_intermediate
```

#### Why this does not count all 384 experts

The router scores all `routed_experts=384`, but it does **not** run all 384 expert MLPs. It runs only `topk=8` routed experts plus `shared_experts=1`. That is the core sparse-MoE saving.

#### Link to the notebook formulas

| Notebook formula | Raw-equation meaning |
| --- | --- |
| `dense_mlp_layer = 6 * H * dense_intermediate` | Layer 0 dense SwiGLU: gate + up + down. |
| `active_moe_layer = 6 * H * moe_intermediate * active_experts` | MoE layers: only 9 active expert SwiGLU MLPs run per token. |
| `router_per_moe_layer = 2 * H * routed_experts` | Router scores all 384 routed experts before top-k selection. |


In [ ]:
dense_mlp_layer = 6 * H * dense_intermediate
active_moe_layer = 6 * H * moe_intermediate * active_experts
router_per_moe_layer = 2 * H * routed_experts

assert dense_mlp_layer == 792_723_456
assert active_moe_layer == 792_723_456
assert router_per_moe_layer == 5_505_024

{
    'dense_mlp_layer': dense_mlp_layer,
    'active_moe_layer': active_moe_layer,
    'router_per_moe_layer': router_per_moe_layer,
}

## 5. Final sum

The final constant is a sum over all text layers:

`61 * attention_projection_per_layer + 1 * dense_mlp_layer + 60 * active_moe_layer + 60 * router_per_moe_layer`

In [ ]:
attention_projection_total = L * attention_projection_per_layer
dense_mlp_total = dense_layers * dense_mlp_layer
active_moe_total = moe_layers * active_moe_layer
router_total = moe_layers * router_per_moe_layer

linear_flops_per_token = (
    attention_projection_total
    + dense_mlp_total
    + active_moe_total
    + router_total
)

assert attention_projection_total == 12_336_889_856
assert dense_mlp_total == 792_723_456
assert active_moe_total == 47_563_407_360
assert router_total == 330_301_440
assert linear_flops_per_token == 61_023_322_112

{
    'attention_projection_total': attention_projection_total,
    'dense_mlp_total': dense_mlp_total,
    'active_moe_total': active_moe_total,
    'router_total': router_total,
    'linear_flops_per_token': linear_flops_per_token,
}

## 6. What this constant does not include

Excluded from this constant:

- Attention scan over context. That is counted by `attention_scan_flops_per_pair` in the pair notebook.
- RMSNorm, RoPE elementwise rotation, softmax, top-k selection, and sampling.
- Communication and memory-transfer cost.
- `lm_head`.
- Quantized bit-operation throughput or byte traffic. This is a logical BF16-equivalent multiply-add FLOP count.

## 7. Source anchors

- HTML source converted from: `z_local/benchserving/mfu_metrics/kimi25_linear_flops_constant.html`.
- Public config source: `moonshotai/Kimi-K2.5/config.json`.
- Local model source: `python/sglang/srt/models/deepseek_v2.py` for `qkv_a`, `q_b`, `kv_b_proj`, `o_proj`, MoE, and router construction.
- Local absorbed MLA source: `python/sglang/srt/models/deepseek_common/attention_forward_methods/forward_mla.py` for `W_KC` and `W_VC` behavior.